# Exploración de la API de World Bank Data360

**Trabajo Práctico Final — Visualización de la Información**

En este notebook recorremos la API de [Data360](https://data360.worldbank.org/en/api) para responder tres preguntas antes de elegir un tema:

1. ¿Qué bases de datos existen?
2. ¿Qué indicadores tiene cada base?
3. ¿Cómo están estructurados los datos (país, tiempo, desagregaciones, temática)?

Todo corre en Colab sin configuración: la API es pública y no pide clave.

In [1]:
# En Colab ya vienen instalados; localmente: pip install pandas requests duckdb altair
import json
import requests
import pandas as pd
import duckdb
import altair as alt

pd.set_option("display.max_colwidth", 80)
BASE = "https://data360api.worldbank.org/data360"


## Endpoints que usamos

| Endpoint | Método | Para qué sirve |
|---|---|---|
| `/searchv2` | POST | Buscador sobre el catálogo (bases e indicadores). Acepta `search`, `filter`, `facets`, `top`, `skip`. |
| `/indicators?datasetId=` | GET | Lista de códigos de indicador de una base. |
| `/metadata` | POST | Ficha completa de un indicador (definición, fuente, metodología). |
| `/disaggregation?datasetId=&indicatorId=` | GET | Valores posibles de cada dimensión (países, sexo, edad…). |
| `/data?DATABASE_ID=&INDICATOR=` | GET | Las observaciones. Paginado de a 1000 filas con `skip`. |
| `/portal/v1/hierarchy?type=country` | GET | Agrupaciones de países: región, nivel de ingreso, tipo de préstamo. |

In [2]:
import time

def pedir(metodo, ruta, intentos=5, **kwargs):
    """Llama a la API reintentando: a veces responde con el cuerpo vacío o corta la conexión."""
    for intento in range(intentos):
        try:
            r = requests.request(metodo, f"{BASE}/{ruta}", timeout=120, **kwargs)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError):
            if intento == intentos - 1:
                raise
            time.sleep(2 * (intento + 1))


def buscar(search="*", filtro=None, top=100, skip=0, facets=None, select=None):
    cuerpo = {"search": search, "top": top, "skip": skip, "count": True}
    if filtro:
        cuerpo["filter"] = filtro
    if facets:
        cuerpo["facets"] = facets
    if select:
        cuerpo["select"] = select
    return pedir("POST", "searchv2", json=cuerpo)


def traer_datos(base, indicador, **filtros):
    """Descarga todas las observaciones de un indicador, recorriendo las páginas."""
    filas, skip = [], 0
    while True:
        params = {"DATABASE_ID": base, "INDICATOR": indicador, "top": 1000, "skip": skip, **filtros}
        pagina = pedir("GET", "data", params=params)
        filas += pagina["value"]
        skip += 1000
        if skip >= pagina["count"]:
            return pd.DataFrame(filas)


## 1. ¿Qué bases de datos existen?

En el catálogo cada base es un documento con `type = 'dataset'`. Las traemos todas paginando el buscador.

In [3]:
bases, skip = [], 0
while True:
    pagina = buscar(filtro="type eq 'dataset'", top=100, skip=skip)
    bases += pagina["value"]
    skip += 100
    if skip >= pagina["@odata.count"]:
        break


def tema_principal(sd):
    temas = sd.get("topics") or []
    return temas[0]["name"] if temas else None


def periodo(sd):
    t = sd.get("time_periods") or []
    return f"{t[0]['start']}–{t[0]['end']}" if t and t[0].get("start") else None


bases = pd.DataFrame([
    {
        "id": b["series_description"]["database_id"],
        "nombre": b["series_description"]["name"],
        "tema": tema_principal(b["series_description"]),
        "indicadores": pd.to_numeric(b["series_description"].get("indicator_count"), errors="coerce"),
        "economias": pd.to_numeric(b["series_description"].get("economies_count"), errors="coerce"),
        "periodo": periodo(b["series_description"]),
        "actualizada": b["series_description"].get("date_last_update"),
    }
    for b in bases
]).drop_duplicates("id").sort_values("indicadores", ascending=False, ignore_index=True)

print(f"{len(bases)} bases de datos en el catálogo")
bases.head(15)


174 bases de datos en el catálogo


,id,nombre,tema,indicadores,economias,periodo,actualizada
0,WB_WDI,World Development Indicators (WDI),Planet,1545,217,1960–2025,2026-06-22
1,IMF_BOP,Balance of Payments (BOP) and International Investment Position (IIP),Prosperity,1146,198,1948–2024,2025-03-07
2,WB_EDSTATS,Education Statistics,People,1072,222,1970–2023,2025-03-07
3,WB_ES,Enterprise Surveys,Prosperity,584,181,2006.0–2026.0,2026-08-25
4,IMF_IFS,International Financial Statistics (IFS),Prosperity,495,206,1940–2024,2025-03-09
5,WB_GS,Gender Statistics,People,363,217,1960–2023,2025-11-04
6,WB_FINDEX,Global Findex Database,Prosperity,280,162,2011–2024,2026-09-24
7,IMF_GFS_SOO,Government Finance Statistics (GFS): Statement of Operations,NaN,218,166,2017–2025,2026-10-01
8,BS_SGI,Sustainable Governance Indicators (SGI),Prosperity,193,41,2014–2024,2026-09-15
9,WB_HNP,Health Nutrition and Population Statistics,People,192,217,1960–2023,2025-03-07


El prefijo del código indica quién produce la base: `WB_` (Banco Mundial), `IMF_` (FMI), `WEF_` (Foro Económico Mundial), `OECD_`, `FAO_`, `ITU_`, `UNICEF_`, `OWID_` (Our World in Data), etc. Data360 funciona como un agregador: reúne bases propias y de terceros con un formato común.

In [4]:
bases["organismo"] = bases["id"].str.split("_").str[0]
duckdb.sql("""
    SELECT organismo,
           count(*)             AS bases,
           sum(indicadores)     AS indicadores
    FROM bases
    GROUP BY organismo
    ORDER BY indicadores DESC NULLS LAST
    LIMIT 12
""").df()


,organismo,bases,indicadores
0,WB,59,5280.0
1,IMF,32,2942.0
2,WEF,3,472.0
3,BS,2,269.0
4,OECD,10,147.0
5,FAO,13,143.0
6,GI,1,114.0
7,VDEM,1,84.0
8,OWID,1,76.0
9,ITU,3,75.0


In [5]:
alt.Chart(bases.dropna(subset=["indicadores"]).head(20)).mark_bar(color="#3b6e8f").encode(
    x=alt.X("indicadores:Q", title="Cantidad de indicadores"),
    y=alt.Y("id:N", sort="-x", title=None),
    tooltip=["nombre", "indicadores", "economias", "periodo"],
).properties(title="Las 20 bases con más indicadores", width=520, height=420)


alt.Chart(...)

In [6]:
bases.groupby("tema", dropna=False).agg(bases=("id", "count"), indicadores=("indicadores", "sum")).sort_values("bases", ascending=False)


,bases,indicadores
tema,,
Prosperity,60,4339
NaN,43,1128
Planet,32,2361
Infrastructure,19,306
Digital,11,110
People,8,1713
International trade,1,21


### Cobertura temporal

Cada base declara el período que cubre. Graficarlo muestra que conviven series largas (desde 1960) con encuestas y rankings que empiezan después de 2010.

In [7]:
cobertura = bases.dropna(subset=["periodo", "indicadores"]).copy()
cobertura[["desde", "hasta"]] = cobertura["periodo"].str.split("–", expand=True).apply(pd.to_numeric, errors="coerce")
cobertura = cobertura.dropna(subset=["desde", "hasta"]).head(30)

alt.Chart(cobertura).mark_bar(height=8, cornerRadius=3, color="#2a78d6").encode(
    x=alt.X("desde:Q", title="Años cubiertos", scale=alt.Scale(zero=False), axis=alt.Axis(format="d")),
    x2="hasta:Q",
    y=alt.Y("id:N", sort=alt.EncodingSortField("desde"), title=None),
    tooltip=["nombre", "periodo", "indicadores", "economias"],
).properties(title="Período cubierto por las 30 bases con más indicadores", width=520, height=460)


alt.Chart(...)

## 2. ¿Qué indicadores tiene cada base?

Con *facets* el buscador nos da directamente cuántos indicadores hay en cada base, sin tener que recorrerlas una por una.

In [8]:
facetas = buscar(filtro="type eq 'indicator'", top=0, facets=["series_description/database_id,count:400"])
por_base = pd.DataFrame(facetas["@search.facets"]["series_description/database_id"]).rename(
    columns={"value": "base", "count": "indicadores"})
print(f"{facetas['@odata.count']:,} indicadores en total, repartidos en {len(por_base)} bases")
por_base.head(10)


9,850 indicadores en total, repartidos en 171 bases


,base,indicadores
0,WB_WDI,1545
1,IMF_BOP,1146
2,WB_EDSTATS,1072
3,WB_ES,584
4,IMF_IFS,495
5,WB_GS,363
6,WB_FINDEX,280
7,IMF_GFS_SOO,218
8,BS_SGI,193
9,WB_HNP,192


Para ver el contenido de una base usamos `/indicators`. Tomamos la más grande, los *World Development Indicators* (WDI), y agrupamos sus códigos por el prefijo temático heredado de la clasificación del Banco Mundial (`SP` población, `SH` salud, `NY` cuentas nacionales, `EN` ambiente, `SE` educación…).

In [9]:
wdi = pedir("GET", "indicators", params={"datasetId": "WB_WDI"})
wdi = pd.Series(wdi, name="codigo").to_frame()
wdi["area"] = wdi["codigo"].str.split("_").str[2]
print(len(wdi), "indicadores en WB_WDI")
wdi["area"].value_counts().head(12)


1499 indicadores en WB_WDI


area
SH    158
SE    156
SL    129
SP     98
NY     94
DT     84
IC     77
NE     65
EN     63
GC     45
TM     45
DC     38
Name: count, dtype: int64

El buscador también sirve para ubicar un concepto en todo el catálogo. Por ejemplo, la esperanza de vida aparece en varias bases (WDI, la base de salud HNP, Gender Statistics…), casi siempre replicando la misma fuente original de Naciones Unidas.

In [10]:
res = buscar("life expectancy at birth", filtro="type eq 'indicator'", top=10,
             select="series_description/idno, series_description/name, series_description/database_id")
pd.json_normalize(res["value"])[["series_description.database_id", "series_description.idno", "series_description.name"]]


,series_description.database_id,series_description.idno,series_description.name
0,WB_WDI,WB_WDI_SP_DYN_LE00_IN,"Life expectancy at birth, total (years)"
1,WB_WDI,WB_WDI_SP_DYN_LE00_FE_IN,"Life expectancy at birth, female (years)"
2,WB_WDI,WB_WDI_SP_DYN_LE00_MA_IN,"Life expectancy at birth, male (years)"
3,WB_GS,WB_GS_SP_DYN_LE00_IN,Life expectancy at birth (years)
4,WB_HNP,WB_HNP_SP_DYN_LE00_IN,"Life expectancy at birth, total (years)"
5,WB_ESG,WB_ESG_SP_DYN_LE00_IN,"Life expectancy at birth, total (years)"
6,WB_HCP,WB_HCP_SP_DYN_LE00_IN,Life expectancy at birth (years)
7,WB_GS,WB_GS_SP_DYN_LE60_IN,Life expectancy at age 60 (years)
8,WB_WDI,WB_WDI_SP_POP_BRTH_MF,Sex ratio at birth (male births per female births)
9,WB_GS,WB_GS_SP_REG_BRTH_ZS,Completeness of birth registration (%)


## 3. ¿Cómo se estructuran los datos?

### 3.1 La ficha del indicador
Cada indicador trae metadatos ricos: definición, unidad, periodicidad, fuente y limitaciones.

In [11]:
meta = pedir("POST", "metadata", json={
    "query": "&$filter=series_description/idno eq 'WB_WDI_SP_DYN_LE00_IN'"})
sd = meta["value"][0]["series_description"]
for campo in ["name", "measurement_unit", "periodicity", "aggregation_method", "definition_long", "limitation"]:
    print(f"{campo}: {str(sd.get(campo))[:300]}\n")
print("fuente:", sd["sources"][0]["name"], "—", sd["sources"][0]["organization"])


name: Life expectancy at birth, total (years)

measurement_unit: Years

periodicity: Annual

aggregation_method: Weighted average

definition_long: Life expectancy at birth indicates the number of years a newborn infant would live if prevailing patterns of mortality at the time of its birth were to stay the same throughout its life.

limitation: Annual data series from United Nations Population Division's World Population Prospects are interpolated data from 5-year period data. Therefore they may not reflect real events as much as observed data.

fuente: World Population Prospects — United Nations (UN)


### 3.2 Las observaciones
Cada fila es una observación con formato tipo SDMX: un valor (`OBS_VALUE`) identificado por un conjunto de dimensiones.

In [12]:
muestra = traer_datos("WB_WDI", "WB_WDI_SP_DYN_LE00_IN", REF_AREA="ARG").sort_values("TIME_PERIOD")
print(muestra.shape)
muestra[["DATABASE_ID", "INDICATOR", "REF_AREA", "TIME_PERIOD", "SEX", "AGE", "URBANISATION", "UNIT_MEASURE", "OBS_VALUE", "OBS_STATUS"]].tail()


(65, 26)


,DATABASE_ID,INDICATOR,REF_AREA,TIME_PERIOD,SEX,AGE,URBANISATION,UNIT_MEASURE,OBS_VALUE,OBS_STATUS
23,WB_WDI,WB_WDI_SP_DYN_LE00_IN,ARG,2020,_T,_T,_T,YR,75.878,A
1,WB_WDI,WB_WDI_SP_DYN_LE00_IN,ARG,2021,_T,_T,_T,YR,73.948,A
51,WB_WDI,WB_WDI_SP_DYN_LE00_IN,ARG,2022,_T,_T,_T,YR,75.806,A
2,WB_WDI,WB_WDI_SP_DYN_LE00_IN,ARG,2023,_T,_T,_T,YR,77.395,A
31,WB_WDI,WB_WDI_SP_DYN_LE00_IN,ARG,2024,_T,_T,_T,YR,77.543,A


| Dimensión | Qué representa |
|---|---|
| `REF_AREA` | País o agregado (ISO3: `ARG`, `CRI`; agregados: `WLD`, `LCN`, `HIC`…) |
| `TIME_PERIOD` | Año (o trimestre/mes según `FREQ`) |
| `SEX`, `AGE`, `URBANISATION` | Desagregaciones; `_T` significa total |
| `COMP_BREAKDOWN_1..3` | Desagregaciones propias de cada base; `_Z` = no aplica |
| `UNIT_MEASURE`, `UNIT_MULT` | Unidad y multiplicador |
| `OBS_STATUS` | Estado de la observación (`A` normal, `E` estimada…) |

`OBS_VALUE` llega como texto, así que hay que convertirlo a número.

In [13]:
dis = pedir("GET", "disaggregation", params={"datasetId": "WB_WDI", "indicatorId": "WB_WDI_SP_DYN_LE00_IN"})
pd.DataFrame([{"dimension": d["field_name"], "valores": len(d["field_value"]), "ejemplo": d["field_value"][:6]} for d in dis])


,dimension,valores,ejemplo
0,FREQ,1,[A]
1,REF_AREA,265,"[ABW, AFE, AFG, AFW, AGO, ALB]"
2,INDICATOR,1,[WB_WDI_SP_DYN_LE00_IN]
3,SEX,1,[_T]
4,AGE,1,[_T]
5,URBANISATION,1,[_T]
6,UNIT_MEASURE,1,[YR]
7,COMP_BREAKDOWN_1,1,[_Z]
8,COMP_BREAKDOWN_2,1,[_Z]
9,COMP_BREAKDOWN_3,1,[_Z]


### 3.3 Países y agrupaciones
`REF_AREA` mezcla países con agregados regionales. La jerarquía del portal permite separarlos y asignar a cada país su región y su grupo de ingreso.

In [14]:
jer = pedir("GET", "portal/v1/hierarchy", params={"type": "country"})["data"]

def aplanar(clave, columna):
    return pd.DataFrame([
        {"iso3": c["id"], "pais": c["name"], columna: g["name"]}
        for g in jer[clave] for c in g["countries"]
    ])

regiones = aplanar("HCL_WB_REGIONS", "region")
ingresos = aplanar("HCL_WB_INCOME", "ingreso")
paises = regiones.merge(ingresos[["iso3", "ingreso"]], on="iso3", how="left")
print(len(paises), "economías con región asignada")
paises.groupby(["region", "ingreso"]).size().unstack(fill_value=0)


220 economías con región asignada


ingreso,High income,Low income,Lower middle income,Upper middle income
region,,,,
East Asia & Pacific,16,1,8,13
Eastern and Southern Africa,1,12,10,3
Europe & Central Asia,40,0,3,15
Latin America & Caribbean,19,0,5,18
"Middle East, North Africa, Afghanistan & Pakistan",8,3,7,5
North America,3,0,0,0
South Asia,0,0,4,2
Western and Central Africa,0,9,10,3


In [15]:
areas = next(d["field_value"] for d in dis if d["field_name"] == "REF_AREA")
agregados = sorted(set(areas) - set(paises["iso3"]))
print(len(agregados), "códigos de REF_AREA que no son países:", agregados[:25], "…")


48 códigos de REF_AREA que no son países: ['AFE', 'AFW', 'ARB', 'CEB', 'CSS', 'EAP', 'EAR', 'EAS', 'ECA', 'ECS', 'EMU', 'EUU', 'FCV', 'HIC', 'HPC', 'IBD', 'IBT', 'IDA', 'IDB', 'IDX', 'LAC', 'LCN', 'LDC', 'LIC', 'LMC'] …


### 3.4 El mismo indicador en varias bases

Vimos que la esperanza de vida aparece en varias bases. ¿Son el mismo dato? Lo comparamos para Argentina.

In [16]:
copias = [("WB_WDI", "WB_WDI_SP_DYN_LE00_IN"), ("WB_HNP", "WB_HNP_SP_DYN_LE00_IN"),
          ("WB_GS", "WB_GS_SP_DYN_LE00_IN"), ("WB_ESG", "WB_ESG_SP_DYN_LE00_IN"), ("WB_HCP", "WB_HCP_SP_DYN_LE00_IN")]
filas = []
for base, codigo in copias:
    d = traer_datos(base, codigo, REF_AREA="ARG")
    total = d[d["SEX"] == "_T"]
    filas.append({
        "base": base,
        "observaciones": len(d),
        "sexo": ", ".join(sorted(d["SEX"].unique())),
        "ultimo_anio": total["TIME_PERIOD"].max(),
        "valor_2021": total.loc[total["TIME_PERIOD"] == "2021", "OBS_VALUE"].squeeze() if (total["TIME_PERIOD"] == "2021").any() else None,
    })
pd.DataFrame(filas)


,base,observaciones,sexo,ultimo_anio,valor_2021
0,WB_WDI,65,_T,2024,73.948
1,WB_HNP,192,"F, M, _T",2023,75.39
2,WB_GS,186,"F, M, _T",2021,75.39
3,WB_ESG,64,_T,2023,73.948
4,WB_HCP,65,_T,2024,74


Tres diferencias importantes:

- **Distinta estructura.** En WDI la esperanza de vida de mujeres y varones son *indicadores separados* (`..._FE_IN`, `..._MA_IN`). En HNP y Gender Statistics es un solo indicador con la dimensión `SEX` (`F`, `M`, `_T`). Por eso hay que filtrar `SEX == "_T"` antes de comparar.
- **Distinta actualización.** Cada base se actualiza en su propio calendario, así que el último año disponible cambia de una a otra.
- **Distintos valores.** Para un mismo país y año los números pueden no coincidir: en 2021, WDI le asigna a Argentina 73,9 años y HNP 75,4. Casi un año y medio de diferencia, porque cada base usa una versión (*vintage*) distinta de las estimaciones de Naciones Unidas o las redondea.

Por eso elegimos **una sola base (WDI)** para todas las variables del análisis: así son consistentes entre sí y están actualizadas.

## Lo que nos llevamos de la exploración

- Data360 reúne **más de 170 bases** de distintos organismos con un **formato único**: misma API, mismas dimensiones y metadatos homogéneos.
- La base más amplia es **WDI** (≈1.500 indicadores, 217 economías, desde 1960). Para un tema transversal es el punto de partida natural.
- Los datos vienen en formato largo (una fila por país–año–desagregación). Hay que filtrar los totales (`_T`), convertir `OBS_VALUE` a número y **excluir los agregados regionales** usando la jerarquía de países.
- Un mismo indicador puede repetirse en varias bases con estructura, cobertura y valores distintos. Conviene elegir una fuente y mantenerla.
- La API a veces devuelve respuestas vacías; conviene envolver las llamadas con reintentos.
- Los metadatos aclaran limitaciones importantes: por ejemplo, la esperanza de vida es una estimación interpolada de Naciones Unidas, no un registro anual directo.

Con esto elegimos el tema: **«Vivir más sin ser más rico»**. Cruzamos esperanza de vida e ingreso por habitante para encontrar los países que se salen del patrón. El análisis sigue en `02_vida_e_ingreso.ipynb`.